# Case-study data intake

This notebook is part of the staged manuscript-reproduction workflow. During automated validation it resolves placeholder paths to the deterministic demo dataset; once local real-data paths are configured it will run against the real case-study artifacts.

In [ ]:
import os
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    env_root = os.environ.get("BSM_PUBLIC_RF_REPO_ROOT")
    if env_root:
        candidate = Path(env_root).resolve()
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate

    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate

    raise RuntimeError("Could not locate the bsm-public-rf repository root.")


REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
from bsm_rfm import build_manuscript_notebook_context, manuscript_runtime_summary_table

context = build_manuscript_notebook_context(REPO_ROOT, "00_case_study_data_intake.ipynb")
summary = manuscript_runtime_summary_table(context)
artifact_shapes = {name: frame.shape for name, frame in context.tables.items()}
summary, artifact_shapes

### Step 1: Load Configuration
    
This cell loads the configuration hierarchy:
1. Read `configs/defaults/manuscript_paths.yml` (read-only defaults)
2. Check for `configs/local/manuscript_paths.local.yml` (user overrides, .gitignore'd)
3. Auto-detect mode: demo if local config missing/invalid, otherwise real data or custom

**Key variables created:**
- `config_dict`: Merged configuration dictionary with all paths and parameters
- `mode`: String indicating 'demo', 'real', or 'custom' execution mode